In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===========================
# 1️⃣ Imports
# ===========================
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# ===========================
# 2️⃣ Device
# ===========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# ===========================
# 2️⃣ Load Dataset
# ===========================
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(ds)

# نحوّل لـ pandas
train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas()

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

print(f"\nTrain size: {train_df.shape}")
print(f"Test size:  {test_df.shape}")
print(f"Label distribution (train):\n{train_df['label'].value_counts()}")
print(f"Label distribution (test):\n{test_df['label'].value_counts()}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Train size: (8530, 2)
Test size:  (1066, 2)
Label distribution (train):
label
1    4265
0    4265
Name: count, dtype: int64
Label distribution (test):
label
1    533
0    533
Name: count, dtype: int64


In [ ]:
# Load model Already saved
model_name="gpt2"
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computing Device: {device}")

Computing Device: cuda


# Data Preprocessing and Tokenization

In [ ]:


# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure padding token is set to EOS token if not defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization to the dataset
tokenized_ds = ds.map(tokenize_function, batched=True)

# Prepare columns for training
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds = tokenized_ds.cast_column("labels", Value('int64')) # Changed to datasets.Value('int64') for classification labels
tokenized_ds.set_format("torch")

print("Data preprocessing completed successfully.")

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8530 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1066 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1066 [00:00<?, ? examples/s]

Data preprocessing completed successfully.


# Model Loading and LoRA Configuration

In [ ]:
# Load the base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id for the model
)

# Configure LoRA parameters specific to GPT-2 architecture
# GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)
# Configure LoRA parameters specific to GPT-2 architecture
# GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc

# Apply LoRA configuration to the model
model = get_peft_model(model, lora_config)

# Display trainable parameters
print("\n--- LoRA Configuration Summary ---")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- LoRA Configuration Summary ---
trainable params: 2,360,832 || all params: 126,802,176 || trainable%: 1.8618


# Training Arguments and Metrics Definition

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/gpt2-rotten-tomatoes/gpt2_sentiment_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    logging_steps=10,
    fp16=False # Changed to False to avoid BFloat16 NotImplementedError
)

# Initialize Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Traing the model

In [ ]:
#Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Execute the training process
print("Initiating training process...")
trainer.train()
print("Training completed.")

Initiating training process...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.441386,0.376482,0.836773,0.839763,0.836773,0.836413
2,0.415705,0.344131,0.849906,0.849906,0.849906,0.849906
3,0.347806,0.352933,0.852720,0.854619,0.852720,0.852523
4,0.307803,0.337113,0.851782,0.851787,0.851782,0.851782
5,0.341488,0.339286,0.853659,0.853838,0.853659,0.853640


Training completed.


# Model Evaluation

In [ ]:
# Perform final evaluation on the test set
print("\n--- Final Evaluation on Test Set ---")
predictions = trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Compute final metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


--- Final Evaluation on Test Set ---


Accuracy:  0.8518
Precision: 0.8518
Recall:    0.8518
F1-Score:  0.8518
